# Linear probing pathology foundation models

Frozen encoder → embeddings → logistic regression → balanced accuracy.

**Train:** NCT-CRC-HE-100K (86 colorectal H&E slides)
**Test:** CRC-VAL-HE-7K (7,180 tiles, 50 *different* patients, different cohort)

The train/test split is across cohorts, not random, so this measures cross-cohort
generalization rather than in-distribution accuracy. That is the property that
matters clinically and the one the robustness literature finds wanting

9 tissue classes: ADI (adipose), BACK (background), DEB (debris), LYM (lymphocytes),
MUC (mucus), MUS (smooth muscle), NORM (normal mucosa), STR (stroma), TUM (tumour epithelium).

**What this notebook does and does not show.** It measures representation quality on
patch classification for one tissue type. It says nothing about slide-level performance,
and it cannot reproduce the medical-centre confounding result, NCT-CRC ships no
per-centre metadata. For that, Camelyon17 (five labelled hospitals) is the dataset.

## 1. Setup

Run once. On Colab, restart the runtime afterwards if `timm` was already loaded.

In [ ]:
# !pip install -q torch torchvision timm transformers scikit-learn umap-learn matplotlib tqdm

## 2. Data

Download both archives from Zenodo record 1214456 and unzip into `./data/`:

- `NCT-CRC-HE-100K-NONORM.zip` → `./data/NCT-CRC-HE-100K-NONORM/`
- `CRC-VAL-HE-7K.zip` → `./data/CRC-VAL-HE-7K/`

Use the **NONORM** version of the training set. The stain-normalised variant has
already had part of the site signature removed, which would quietly flatter the results.

Both unzip into class-named subfolders, which is the layout `ImageFolder` expects.
The 100K archive is roughly 8 GB — start it before you do anything else.

In [ ]:
TRAIN_DIR = "./data/NCT-CRC-HE-100K-NONORM"
TEST_DIR  = "./data/CRC-VAL-HE-7K"

# Training tiles sampled per class. Start at 200 to verify the pipeline,
# then raise. 500 is enough for a stable linear probe.
PER_CLASS  = 500
BATCH_SIZE = 64
NUM_WORKERS = 4

In [ ]:
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
print("device:", DEVICE)

assert os.path.isdir(TRAIN_DIR), f"missing {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"missing {TEST_DIR}"
print("train classes:", sorted(os.listdir(TRAIN_DIR))[:12])

## 3. Hugging Face access

`resnet50` and `phikon` work immediately. `uni` and `h-optimus-0` are gated — request
access on their model pages and log in below. Approval can take hours or days, so
request it early and run the ungated models meanwhile.

In [ ]:
# from huggingface_hub import login
# login()   # paste a token with read access

## 4. Encoders

Each entry returns `(model, transform, embedding_dim)`. Note that the normalisation
constants differ per model — H-optimus-0 in particular does **not** use ImageNet
statistics, and using the wrong ones silently degrades its numbers rather than erroring.

In [ ]:
def build_encoder(name):
    if name == "resnet50":
        import timm
        m = timm.create_model("resnet50", pretrained=True, num_classes=0)
        tf = transforms.Compose([
            transforms.Resize(224), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        return m, tf, 2048

    if name == "phikon":
        from transformers import AutoModel
        hf = AutoModel.from_pretrained("owkin/phikon")

        class Wrap(nn.Module):
            def __init__(self, hf):
                super().__init__(); self.hf = hf
            def forward(self, x):
                # Phikon uses the CLS token as the image representation.
                return self.hf(pixel_values=x).last_hidden_state[:, 0, :]

        tf = transforms.Compose([
            transforms.Resize(224), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        return Wrap(hf), tf, 768

    if name == "uni":
        import timm
        m = timm.create_model("hf-hub:MahmoodLab/UNI", pretrained=True,
                              init_values=1e-5, dynamic_img_size=True)
        tf = transforms.Compose([
            transforms.Resize(224), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        return m, tf, 1024

    if name == "h-optimus-0":
        import timm
        m = timm.create_model("hf-hub:bioptimus/H-optimus-0", pretrained=True,
                              init_values=1e-5, dynamic_img_size=False)
        tf = transforms.Compose([
            transforms.Resize(224), transforms.ToTensor(),
            transforms.Normalize(mean=(0.707223, 0.578729, 0.703617),
                                 std=(0.211883, 0.230117, 0.177517))])
        return m, tf, 1536

    raise ValueError(f"unknown model: {name}")

## 5. Feature extraction

Embeddings are cached to `features/{model}.npz`, so re-running the probe with
different hyperparameters does not re-run the encoder.

In [ ]:
def subsample(ds, per_class, seed=0):
    """At most `per_class` images from each class, deterministically."""
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    keep = []
    for c in np.unique(targets):
        idx = np.flatnonzero(targets == c)
        if len(idx) > per_class:
            idx = rng.choice(idx, per_class, replace=False)
        keep.extend(idx.tolist())
    return Subset(ds, sorted(keep))


@torch.no_grad()
def extract(model, loader):
    model.eval().to(DEVICE)
    feats, labels = [], []
    for x, y in tqdm(loader, leave=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            f = model(x.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


def get_features(name, force=False):
    os.makedirs("features", exist_ok=True)
    path = f"features/{name}.npz"
    if os.path.exists(path) and not force:
        d = np.load(path)
        print(f"{name}: loaded cache")
        return d["Xtr"], d["ytr"], d["Xte"], d["yte"], list(d["classes"])

    model, tf, dim = build_encoder(name)
    train_ds = subsample(datasets.ImageFolder(TRAIN_DIR, tf), PER_CLASS)
    test_ds  = datasets.ImageFolder(TEST_DIR, tf)
    classes  = test_ds.classes
    print(f"{name}: dim={dim} train={len(train_ds)} test={len(test_ds)}")

    mk = lambda ds: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    Xtr, ytr = extract(model, mk(train_ds))
    Xte, yte = extract(model, mk(test_ds))

    np.savez_compressed(path, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, classes=classes)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return Xtr, ytr, Xte, yte, classes

## 6. The probe

L2-normalise before fitting. This is standard for frozen-feature evaluation and it
stops differences in embedding magnitude between models from confounding the comparison.

In [ ]:
def linear_probe(Xtr, ytr, Xte, yte, classes, name="", verbose=True):
    n = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True)
    clf = LogisticRegression(max_iter=3000, C=1.0, n_jobs=-1)
    clf.fit(n(Xtr), ytr)
    pred = clf.predict(n(Xte))
    bal = balanced_accuracy_score(yte, pred)
    if verbose:
        print(f"\n=== {name} — balanced accuracy: {bal:.4f} ===\n")
        print(classification_report(yte, pred, target_names=classes, digits=3))
    return bal, pred

## 7. Run

Start with `resnet50` at `PER_CLASS = 200` to confirm the pipeline. Then raise
`PER_CLASS` and add the pathology models.

In [ ]:
MODELS = ["resnet50", "phikon"]        # add "uni", "h-optimus-0" once access is granted

results = {}
for name in MODELS:
    Xtr, ytr, Xte, yte, classes = get_features(name)
    bal, pred = linear_probe(Xtr, ytr, Xte, yte, classes, name=name)
    results[name] = {"balanced_accuracy": bal, "pred": pred, "yte": yte, "classes": classes}

In [ ]:
import pandas as pd
pd.DataFrame([{"model": k, "balanced_accuracy": round(v["balanced_accuracy"], 4)}
              for k, v in results.items()]).sort_values("balanced_accuracy", ascending=False)

## 8. Per-class comparison

**Read this before the headline number.** BACK and ADI are near-trivial and inflate the
mean. The clinically meaningful distinctions are TUM vs STR vs NORM. A model that wins
overall but loses on TUM is not the better model for a diagnostic task.

In [ ]:
from sklearn.metrics import f1_score

rows = []
for name, r in results.items():
    f1 = f1_score(r["yte"], r["pred"], average=None)
    rows.append({"model": name, **{c: round(f, 3) for c, f in zip(r["classes"], f1)}})
pd.DataFrame(rows).set_index("model")

## 9. Confusion matrices

Which pairs get confused matters more than how many. Systematic TUM↔STR confusion
means something different from errors scattered across all classes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, len(results), figsize=(6.5 * len(results), 5.5))
axes = np.atleast_1d(axes)
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["yte"], r["pred"], normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=r["classes"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"{name} (row-normalised)")
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 10. UMAP of the embedding space

Qualitative, but it makes the difference between representations visible in a way a
single number does not. Well-separated tissue clusters mean the encoder has learned
morphology; blurred or interleaved clusters mean it has not.

In [ ]:
import umap

MODEL_TO_PLOT = MODELS[-1]
d = np.load(f"features/{MODEL_TO_PLOT}.npz")
Xte, yte, classes = d["Xte"], d["yte"], list(d["classes"])
Xte = Xte / np.linalg.norm(Xte, axis=1, keepdims=True)

emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Xte)

plt.figure(figsize=(8, 7))
for i, c in enumerate(classes):
    m = yte == i
    plt.scatter(emb[m, 0], emb[m, 1], s=2, alpha=0.6, label=c)
plt.legend(markerscale=6, fontsize=9)
plt.title(f"UMAP — {MODEL_TO_PLOT} embeddings, CRC-VAL-HE-7K")
plt.tight_layout(); plt.show()

## 11. Notes for the meeting

Fill these in as you go — they are more useful than the numbers themselves.

- Did pathology pretraining beat ImageNet ResNet-50, and by how much? If it did not,
  suspect the transform or the normalisation constants before believing the result.
- Which classes did each model fail on? Was the ranking on TUM the same as overall?
- Did model size track performance, or did the smaller pathology models hold up?
- What broke, and how long did each step actually take? Practical friction is worth
  reporting — it tells your supervisor what infrastructure the group needs.

**Caveat to state plainly:** one tissue type, patch-level, no per-centre metadata.
This is a tooling exercise with an honest cross-cohort split, not a benchmark result.